In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
import os

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, early_stopping

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [2]:
train = pd.read_csv("train-2.csv")
test = pd.read_csv("test-2.csv")
train["Date"] = pd.to_datetime(train["Date"])
test["Date"] = pd.to_datetime(test["Date"])



print("\nTrain date:")
print(train["Date"].min(), "->", train["Date"].max())

print("\nTest date:")
print(test["Date"].min(), "->", test["Date"].max())

train.shape, train.dtypes

test.shape, test.dtypes


Train date:
2022-01-01 00:00:00 -> 2023-10-31 00:00:00

Test date:
2023-12-01 00:00:00 -> 2024-01-01 00:00:00


((3200, 40),
 Date                       datetime64[us]
 Store ID                              str
 Product ID                            str
 Inventory Level                     int64
 Units Sold                          int64
 Units Ordered                       int64
 Price                             float64
 Discount                            int64
 Holiday/Promotion                   int64
 Year                                int64
 Month                               int64
 Day                                 int64
 DayOfWeek                           int64
 NET_PRICE                         float64
 inventory_lag_1                   float64
 is_out_of_stock_lag_1               int64
 Price_Diff                        float64
 Price_Ratio                       float64
 units_sold_lag_1                  float64
 units_sold_lag_2                  float64
 units_sold_lag_3                  float64
 units_sold_lag_7                  float64
 units_sold_lag_14                 float6

In [3]:
train = train.sort_values(["Store ID", "Product ID", "Date"]).reset_index(drop=True)
test = test.sort_values(["Store ID", "Product ID", "Date"]).reset_index(drop=True)



In [4]:
train.head()

,Date,Store ID,Product ID,Inventory Level,Units Sold,Units Ordered,Price,Discount,Holiday/Promotion,Year,Month,Day,DayOfWeek,NET_PRICE,inventory_lag_1,is_out_of_stock_lag_1,Price_Diff,Price_Ratio,units_sold_lag_1,units_sold_lag_2,units_sold_lag_3,units_sold_lag_7,units_sold_lag_14,units_sold_lag_30,units_sold_lag_365,sales_roll_mean_7,sales_roll_mean_30,Category_Electronics,Category_Furniture,Category_Groceries,Category_Toys,Region_North,Region_South,Region_West,Weather Condition_Rainy,Weather Condition_Snowy,Weather Condition_Sunny,Seasonality_NEW_Spring,Seasonality_NEW_Summer,Seasonality_NEW_Winter
0,2022-01-01,S001,P0001,231,127,55,33.50,20,0,2022,1,1,5,26.800,NaN,0,3.81,1.128326,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0,1,0,0,1,0,0,0,0,1
1,2022-01-02,S001,P0001,116,81,104,27.95,10,0,2022,1,2,6,25.155,231.0,0,-2.94,0.904824,127.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0,0,0,1,0,0,0,0,0,1
2,2022-01-03,S001,P0001,154,5,189,62.70,20,0,2022,1,3,0,50.160,116.0,0,4.48,1.076950,81.0,127.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,1,1,0,0,0,0,1
3,2022-01-04,S001,P0001,85,58,193,77.88,15,1,2022,1,4,1,66.198,154.0,0,1.89,1.024872,5.0,81.0,127.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0,0,1,0,0,0,0,0,0,1
4,2022-01-05,S001,P0001,238,147,37,28.46,20,1,2022,1,5,2,22.768,85.0,0,-0.94,0.968027,58.0,5.0,81.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0,0,1,0,0,0,1,0,0,1


In [5]:
test.head()

,Date,Store ID,Product ID,Inventory Level,Units Sold,Units Ordered,Price,Discount,Holiday/Promotion,Year,Month,Day,DayOfWeek,NET_PRICE,inventory_lag_1,is_out_of_stock_lag_1,Price_Diff,Price_Ratio,units_sold_lag_1,units_sold_lag_2,units_sold_lag_3,units_sold_lag_7,units_sold_lag_14,units_sold_lag_30,units_sold_lag_365,sales_roll_mean_7,sales_roll_mean_30,Category_Electronics,Category_Furniture,Category_Groceries,Category_Toys,Region_North,Region_South,Region_West,Weather Condition_Rainy,Weather Condition_Snowy,Weather Condition_Sunny,Seasonality_NEW_Spring,Seasonality_NEW_Summer,Seasonality_NEW_Winter
0,2023-12-01,S001,P0001,397,185,58,59.79,10,1,2023,12,1,4,53.811,417.0,0,-2.88,0.954045,244.0,110.0,89.0,16.0,416.0,15.0,191.0,152.714286,137.700000,0,0,1,0,0,0,1,0,0,0,0,0,0
1,2023-12-02,S001,P0001,214,200,136,60.90,20,0,2023,12,2,5,48.720,397.0,0,0.46,1.007611,185.0,244.0,110.0,371.0,223.0,139.0,289.0,176.857143,143.366667,0,0,0,1,0,0,0,0,0,0,0,0,0
2,2023-12-03,S001,P0001,482,122,81,21.92,0,0,2023,12,3,6,21.920,214.0,0,0.04,1.001828,200.0,185.0,244.0,52.0,105.0,136.0,7.0,152.428571,145.400000,0,0,1,0,0,1,0,0,0,1,0,0,0
3,2023-12-04,S001,P0001,57,49,26,71.24,0,0,2023,12,4,0,71.240,482.0,0,-4.70,0.938109,122.0,200.0,185.0,187.0,365.0,180.0,350.0,162.428571,144.933333,0,0,0,0,0,1,0,1,0,0,0,0,0
4,2023-12-05,S001,P0001,104,66,122,44.77,0,1,2023,12,5,1,44.770,57.0,0,-3.39,0.929610,49.0,122.0,200.0,89.0,21.0,21.0,6.0,142.714286,140.566667,1,0,0,0,1,0,0,0,0,1,0,0,0


In [6]:
target = "Units Sold"

y_train = train[target]
y_test = test[target]

In [7]:
all_data = pd.concat([train, test], ignore_index=True)
all_data = all_data.sort_values(["Store ID", "Product ID", "Date"]).reset_index(drop=True)

all_data["sales_ewm_7"] = all_data.groupby(["Store ID", "Product ID"])["Units Sold"].transform(
    lambda x: x.shift(1).ewm(span=7, adjust=False).mean())
all_data["sales_ewm_30"] = all_data.groupby(["Store ID", "Product ID"])["Units Sold"].transform(
    lambda x: x.shift(1).ewm(span=30, adjust=False).mean())

train_end_date = train["Date"].max()
train = all_data[all_data["Date"] <= train_end_date].copy()
test = all_data[all_data["Date"] > train_end_date].copy()

In [8]:
drop_cols = [
    "Units Sold", "Date",
    "Units Ordered",     # leakage
    "Inventory Level",   # leakage - o gunun kapanis stogu, tahmin aninda bilinmez
]
for maybe_leak in ["Demand Forecast", "Demand_Forecast_Baseline",
                    "Store ID_Code", "Product ID_Code", "Category_Code"]:
    if maybe_leak in train.columns:
        drop_cols.append(maybe_leak)
print("drop_cols:", drop_cols)
# NOT: inventory_lag_1 ve is_out_of_stock_lag_1 (varsa) drop_cols'ta DEGIL --
# bunlar bir onceki gunun bilgisi, tahmin aninda mesru sekilde bilinir.

X_train = train.drop(columns=drop_cols)
y_train = train[target]
X_test = test.drop(columns=drop_cols)
y_test = test[target]

categorical_cols = [
    c for c in X_train.columns
    if pd.api.types.is_string_dtype(X_train[c]) or isinstance(X_train[c].dtype, pd.CategoricalDtype)
]
print("categorical_cols:", categorical_cols)
for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

drop_cols: ['Units Sold', 'Date', 'Units Ordered', 'Inventory Level']
categorical_cols: ['Store ID', 'Product ID']


In [9]:
val_baseline = train.sort_values(["Store ID", "Product ID", "Date"]).reset_index(drop=True)
val_baseline["naive_prediction"] = val_baseline.groupby(["Store ID", "Product ID"])["Units Sold"].shift(1)
val_baseline["ma_7_prediction"] = val_baseline.groupby(["Store ID", "Product ID"])["Units Sold"].transform(
    lambda x: x.shift(1).rolling(7).mean())
val_baseline["ewm_7_prediction"] = val_baseline.groupby(["Store ID", "Product ID"])["Units Sold"].transform(
    lambda x: x.shift(1).ewm(span=7, adjust=False).mean())

validation_days = 30
val_start_date = train["Date"].max() - pd.Timedelta(days=validation_days - 1)
val_baseline_eval = val_baseline[val_baseline["Date"] >= val_start_date].copy()

In [10]:
def mae(y_true, y_pred): return mean_absolute_error(y_true, y_pred)
def rmse(y_true, y_pred): return np.sqrt(mean_squared_error(y_true, y_pred))
def smape(y_true, y_pred):
    denom = np.abs(y_true) + np.abs(y_pred)
    return np.mean(2 * np.abs(y_true - y_pred) / np.where(denom == 0, 1, denom)) * 100
def wape(y_true, y_pred):
    return (np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))) * 100
def evaluate_model(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "sMAPE": smape(y_true, y_pred), "WAPE": wape(y_true, y_pred), "R2": r2_score(y_true, y_pred)}

# DUZELTME: early stopping'i artik MAE degil, dogrudan SMAPE'e gore yapmak icin.
# ONEMLI: LGBMRegressor.fit(eval_metric=...) -- yani SKLEARN wrapper -- fonksiyonu
# dogrudan (y_true, y_pred) ile cagirir; Dataset nesnesi/get_label() YOK.
# (native lgb.train(feval=...) kullansaydik (preds, train_data) beklerdik, ama
# burada LGBMRegressor kullandigimiz icin sklearn imzasi gerekiyor.)
def lgbm_smape_raw(y_true, y_pred):
    return "SMAPE", smape(y_true, y_pred), False

def lgbm_smape(y_true, y_pred):
    # log1p uzayindaki modeller icin -- gercek SMAPE'i gormek adina expm1 ile geri donusturur
    return "SMAPE", smape(np.expm1(y_true), np.expm1(y_pred)), False

def xgb_smape(y_true, y_pred):
    return smape(y_true, y_pred)

naive_eval = val_baseline_eval.dropna(subset=["naive_prediction"])
naive_results = evaluate_model(naive_eval["Units Sold"], naive_eval["naive_prediction"])
ma_eval = val_baseline_eval.dropna(subset=["ma_7_prediction"])
ma_results = evaluate_model(ma_eval["Units Sold"], ma_eval["ma_7_prediction"])
ewm_eval = val_baseline_eval.dropna(subset=["ewm_7_prediction"])
ewm_results = evaluate_model(ewm_eval["Units Sold"], ewm_eval["ewm_7_prediction"])
baseline_results = pd.DataFrame({"Naive": naive_results, "Moving Average 7": ma_results, "EWM 7": ewm_results}).T
print("\nBaseline:\n", baseline_results)


Baseline:
                          MAE        RMSE      sMAPE       WAPE        R2
Naive             119.925333  153.795774  92.584841  87.444402 -0.996421
Moving Average 7   93.197238  115.934272  73.511822  67.955423 -0.134454
EWM 7              93.265247  116.173594  73.454158  68.005012 -0.139143


In [11]:
validation_start = train["Date"].max() - pd.Timedelta(days=validation_days - 1)
train_model = train[train["Date"] < validation_start].copy()
validation = train[train["Date"] >= validation_start].copy()

X_tr = train_model.drop(columns=drop_cols)
y_tr = train_model[target]
X_val = validation.drop(columns=drop_cols)
y_val = validation[target]
for col in categorical_cols:
    X_tr[col] = X_tr[col].astype("category")
    X_val[col] = X_val[col].astype("category")

In [12]:
# DUZELTME: learning_rate 0.05 -> 0.02, max_depth 6 -> 5 (daha temkinli, ezberlemeyi azaltmak icin)
# DUZELTME: eval_metric artik MAE degil, dogrudan SMAPE (xgb_smape) -- early stopping bu metrige gore calisiyor
# DUZELTME: early_stopping_rounds 50 -> 200 (04'teki gibi, cok erken durmayi engellemek icin)
xgb_model = XGBRegressor(
    objective="reg:squarederror", n_estimators=1000, learning_rate=0.02, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
    enable_categorical=True, early_stopping_rounds=200, eval_metric=xgb_smape,
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
xgb_val_pred = xgb_model.predict(X_val)
xgb_val_results = evaluate_model(y_val, xgb_val_pred)
print("\nXGBoost best_iteration:", xgb_model.best_iteration, "| val:", xgb_val_results)


XGBoost best_iteration: 11 | val: {'MAE': 88.07649993896484, 'RMSE': 108.85365031775461, 'sMAPE': 70.79068612100586, 'WAPE': 64.22160013666212, 'R2': -0.00011348724365234375}


In [13]:
# DUZELTME: learning_rate 0.05 -> 0.02, num_leaves 31 -> 10 (04'teki gibi daha sade model)
# DUZELTME: feval=lgbm_smape_raw ile early stopping artik SMAPE'e gore (MAE degil)
# DUZELTME: early_stopping_rounds 50 -> 200
lgbm_model = LGBMRegressor(
    objective="regression", n_estimators=1000, learning_rate=0.02, num_leaves=10, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1,
)
lgbm_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
               eval_metric=lgbm_smape_raw,
               callbacks=[early_stopping(stopping_rounds=200, verbose=False)])
lgbm_val_pred = lgbm_model.predict(X_val)
lgbm_val_results = evaluate_model(y_val, lgbm_val_pred)
print("LightGBM best_iteration:", lgbm_model.best_iteration_, "| val:", lgbm_val_results)

/opt/anaconda3/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM best_iteration: 1 | val: {'MAE': 88.11516050686342, 'RMSE': 108.85326881728798, 'sMAPE': 70.80391206103799, 'WAPE': 64.24979012930147, 'R2': -0.00010643862315640185}


In [14]:
y_tr_log = np.log1p(y_tr)
y_val_log = np.log1p(y_val)

# DUZELTME: ayni temkinli hiperparametreler + SMAPE'e gore early stopping (log->expm1 donusumlu)
xgb_log_model = XGBRegressor(
    objective="reg:squarederror", n_estimators=1000, learning_rate=0.02, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
    enable_categorical=True, early_stopping_rounds=200,
    eval_metric=lambda yt, yp: smape(np.expm1(yt), np.expm1(yp)),
)
xgb_log_model.fit(X_tr, y_tr_log, eval_set=[(X_val, y_val_log)], verbose=False)
xgb_log_pred = np.expm1(xgb_log_model.predict(X_val))
xgb_log_results = evaluate_model(y_val, xgb_log_pred)

lgbm_log_model = LGBMRegressor(
    objective="regression", n_estimators=1000, learning_rate=0.02, num_leaves=10, max_depth=5,
    random_state=42, n_jobs=-1, verbose=-1,
)
lgbm_log_model.fit(
    X_tr, y_tr_log,
    eval_set=[(X_val, y_val_log)],
    eval_metric=lgbm_smape,   # DUZELTME: log uzayindan expm1 ile geri donduren SMAPE feval
    categorical_feature=categorical_cols,
    callbacks=[early_stopping(stopping_rounds=200, verbose=False)],
)
lgbm_log_pred = np.expm1(lgbm_log_model.predict(X_val))
lgbm_log_results = evaluate_model(y_val, lgbm_log_pred)

all_validation_results = pd.DataFrame({
    "XGBoost": xgb_val_results, "XGBoost + Log": xgb_log_results,
    "LightGBM": lgbm_val_results, "LightGBM + Log": lgbm_log_results,
}).T
final_comparison = pd.concat([baseline_results, all_validation_results])
print("\nTam kiyaslama:\n", final_comparison)

/opt/anaconda3/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



Tam kiyaslama:
                          MAE        RMSE      sMAPE       WAPE        R2
Naive             119.925333  153.795774  92.584841  87.444402 -0.996421
Moving Average 7   93.197238  115.934272  73.511822  67.955423 -0.134454
EWM 7              93.265247  116.173594  73.454158  68.005012 -0.139143
XGBoost            88.076500  108.853650  70.790686  64.221600 -0.000113
XGBoost + Log      87.363022  119.578675  73.508433  63.701356 -0.206899
LightGBM           88.115161  108.853269  70.803912  64.249790 -0.000106
LightGBM + Log     87.358914  119.585280  73.506080  63.698368 -0.207032


In [15]:
# DUZELTME: arama araligi artik temkinli degerler etrafinda (0.05 yerine 0.02 civari, num_leaves 10-20)
tscv = TimeSeriesSplit(n_splits=3)
lgbm_param_grid = {"n_estimators": [300, 500, 800], "learning_rate": [0.02], "num_leaves": [10, 20]}
base_lgbm = LGBMRegressor(objective="regression", random_state=42, n_jobs=-1, verbose=-1)
lgbm_grid_search = GridSearchCV(base_lgbm, lgbm_param_grid, cv=tscv,
                                 scoring="neg_mean_absolute_error", n_jobs=-1, verbose=0)
lgbm_grid_search.fit(X_tr, y_tr)
best_lgbm_model = lgbm_grid_search.best_estimator_
best_lgbm_val_pred = best_lgbm_model.predict(X_val)
best_lgbm_val_results = evaluate_model(y_val, best_lgbm_val_pred)
print("\nEn iyi parametreler:", lgbm_grid_search.best_params_)
final_comparison.loc["LightGBM (Tuned)"] = best_lgbm_val_results
print("\nGuncel tam kiyaslama:\n", final_comparison)


En iyi parametreler: {'learning_rate': 0.02, 'n_estimators': 300, 'num_leaves': 10}

Guncel tam kiyaslama:
                          MAE        RMSE      sMAPE       WAPE        R2
Naive             119.925333  153.795774  92.584841  87.444402 -0.996421
Moving Average 7   93.197238  115.934272  73.511822  67.955423 -0.134454
EWM 7              93.265247  116.173594  73.454158  68.005012 -0.139143
XGBoost            88.076500  108.853650  70.790686  64.221600 -0.000113
XGBoost + Log      87.363022  119.578675  73.508433  63.701356 -0.206899
LightGBM           88.115161  108.853269  70.803912  64.249790 -0.000106
LightGBM + Log     87.358914  119.585280  73.506080  63.698368 -0.207032
LightGBM (Tuned)   88.108642  108.929147  70.821839  64.245037 -0.001501


In [16]:
lgbm_importance = pd.DataFrame({
    "Feature": X_tr.columns, "Importance": lgbm_model.feature_importances_
}).sort_values("Importance", ascending=False)
print("\nLightGBM top 15:\n", lgbm_importance.head(15))


LightGBM top 15:
                     Feature  Importance
19        units_sold_lag_30           2
1                Product ID           2
14         units_sold_lag_1           2
17         units_sold_lag_7           1
7                       Day           1
12               Price_Diff           1
28             Region_South           0
24       Category_Furniture           0
25       Category_Groceries           0
26            Category_Toys           0
27             Region_North           0
29              Region_West           0
22       sales_roll_mean_30           0
30  Weather Condition_Rainy           0
31  Weather Condition_Snowy           0


In [17]:
candidate_models = {
    "XGBoost": xgb_model, "XGBoost + Log": xgb_log_model,
    "LightGBM": lgbm_model, "LightGBM + Log": lgbm_log_model,
    "LightGBM (Tuned)": best_lgbm_model,
}

# DUZELTME: secim artik R2 (idxmax) degil, sMAPE (idxmin) bazli -- projenin hedef metrigi SMAPE oldugu icin
ml_candidates = final_comparison.loc[list(candidate_models.keys())]
best_model_name = ml_candidates["sMAPE"].idxmin()
best_model = candidate_models[best_model_name]
is_log_model = "Log" in best_model_name
print(f"\n>>> Secilen final model: {best_model_name} "
      f"(Validation sMAPE: {ml_candidates.loc[best_model_name, 'sMAPE']:.2f}%, "
      f"WAPE: {ml_candidates.loc[best_model_name, 'WAPE']:.2f}%)")
print(">>> NOT: Tum adaylarin R2'si ~0 -- bu, hicbir modelin gercekte guclu bir "
      "sinyal yakalamadigi anlamina gelir (bkz. asagidaki yorum).")


>>> Secilen final model: XGBoost (Validation sMAPE: 70.79%, WAPE: 64.22%)
>>> NOT: Tum adaylarin R2'si ~0 -- bu, hicbir modelin gercekte guclu bir sinyal yakalamadigi anlamina gelir (bkz. asagidaki yorum).


In [18]:
if is_log_model:
    y_train_for_final = np.log1p(y_train)
else:
    y_train_for_final = y_train

final_params = best_model.get_params()
if best_model_name.startswith("LightGBM"):
    final_params.pop("importance_type", None)
    final_model = LGBMRegressor(**final_params)
    final_model.fit(X_train, y_train_for_final, categorical_feature=categorical_cols)
else:
    final_params.pop("early_stopping_rounds", None)
    final_model = XGBRegressor(**final_params)
    final_model.fit(X_train, y_train_for_final)

raw_test_pred = final_model.predict(X_test)
test_pred = np.clip(np.expm1(raw_test_pred) if is_log_model else raw_test_pred, 0, None)
test_results = evaluate_model(y_test, test_pred)
print("\n>>> GERCEK TEST SETI sonuclari:\n", test_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
resid = y_test.values - test_pred
axes[0].hist(resid, bins=30, color="#1f77b4", edgecolor="black", alpha=0.7)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title("Test Kalinti Dagilimi")
axes[1].scatter(y_test, test_pred, alpha=0.4, s=15, color="#2ca02c")
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[1].set_title("Test: Gercek vs Tahmin")
plt.tight_layout()
plt.savefig("final_test_evaluation.png")
plt.close()


>>> GERCEK TEST SETI sonuclari:
 {'MAE': 87.18468475341797, 'RMSE': 106.21324272277681, 'sMAPE': 71.80579741076879, 'WAPE': 64.59380711125971, 'R2': -0.009291529655456543}


In [19]:
def recursive_forecast(model, is_log, history_df, categorical_cols, drop_cols,
                        feature_cols, n_days=14, target_col="Units Sold"):
   
    history_df = history_df.sort_values(["Store ID", "Product ID", "Date"]).copy()
    last_date = history_df["Date"].max()
    future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=n_days)

    forecasts = []
    for (store, product), group in history_df.groupby(["Store ID", "Product ID"]):
        group = group.sort_values("Date").reset_index(drop=True)
        series = group[target_col].tolist()
        last_row = group.iloc[-1]

        for d in future_dates:
            feat = {c: last_row[c] for c in feature_cols}
            feat["Store ID"], feat["Product ID"] = store, product
            if "Year" in feat: feat["Year"] = d.year
            if "Month" in feat: feat["Month"] = d.month
            if "Day" in feat: feat["Day"] = d.day
            if "DayOfWeek" in feat: feat["DayOfWeek"] = d.dayofweek

            for lag in [1, 2, 3, 7, 14, 30, 365]:
                col = f"units_sold_lag_{lag}"
                if col in feat:
                    feat[col] = series[-lag] if len(series) >= lag else np.nan
            if "sales_roll_mean_7" in feat:
                feat["sales_roll_mean_7"] = np.mean(series[-7:])
            if "sales_roll_mean_30" in feat:
                feat["sales_roll_mean_30"] = np.mean(series[-30:])
            if "sales_ewm_7" in feat:
                feat["sales_ewm_7"] = pd.Series(series).ewm(span=7, adjust=False).mean().iloc[-1]
            if "sales_ewm_30" in feat:
                feat["sales_ewm_30"] = pd.Series(series).ewm(span=30, adjust=False).mean().iloc[-1]

            X_future = pd.DataFrame([feat])[feature_cols]
            for col in categorical_cols:
                if col in X_future:
                    X_future[col] = X_future[col].astype("category")

            raw_pred = model.predict(X_future)[0]
            pred = max(np.expm1(raw_pred) if is_log else raw_pred, 0)
            series.append(pred)

            forecasts.append({"Date": d, "Store ID": store, "Product ID": product,
                               "predicted_units_sold": round(pred, 1)})
    return pd.DataFrame(forecasts)


future_forecast = recursive_forecast(final_model, is_log_model, train,
                                      categorical_cols, drop_cols,
                                      feature_cols=X_train.columns.tolist(), n_days=14)
print("\nGelecek tahmini ornegi:\n", future_forecast.head(10))


Gelecek tahmini ornegi:
         Date Store ID Product ID  predicted_units_sold
0 2023-11-01     S001      P0001            145.399994
1 2023-11-02     S001      P0001            152.699997
2 2023-11-03     S001      P0001            142.199997
3 2023-11-04     S001      P0001            142.600006
4 2023-11-05     S001      P0001            149.399994
5 2023-11-06     S001      P0001            140.600006
6 2023-11-07     S001      P0001            146.300003
7 2023-11-08     S001      P0001            141.399994
8 2023-11-09     S001      P0001            147.300003
9 2023-11-10     S001      P0001            137.100006


In [20]:
future_forecast.to_csv("future_demand_forecast.csv", index=False)
final_comparison.to_csv("model_comparison_results.csv")

model_artifacts = {
    "model": final_model, "features": X_train.columns.tolist(),
    "categorical_cols": categorical_cols, "is_log_model": is_log_model,
    "sigma_error": float((y_val - (np.expm1(best_model.predict(X_val)) if is_log_model
                                    else best_model.predict(X_val))).std()),
}
os.makedirs("models", exist_ok=True)
joblib.dump(model_artifacts, "models/demand_forecast_model.pkl")

['models/demand_forecast_model.pkl']

In [21]:
all_df = pd.concat([train, test], ignore_index=True).sort_values(
    ["Store ID", "Product ID", "Date"]).reset_index(drop=True)
X_all = all_df[model_artifacts["features"]].copy()
for col in categorical_cols:
    X_all[col] = X_all[col].astype("category")
raw_all_pred = final_model.predict(X_all)
all_df["predicted_demand"] = np.clip(
    np.expm1(raw_all_pred) if is_log_model else raw_all_pred, 0, None)

inventory_df = all_df.rename(columns={
    "Date": "date", "Store ID": "store", "Product ID": "product",
    "Units Sold": "units_sold", "Inventory Level": "inventory_level",
})[["date", "store", "product", "units_sold", "inventory_level", "predicted_demand"]]
inventory_df.to_csv("envanter_verisi.csv", index=False)

print("\n>>> Kaydedildi: future_demand_forecast.csv, models/demand_forecast_model.pkl, "
      "model_comparison_results.csv, envanter_verisi.csv")
print("\nTUM ADIMLAR TAMAMLANDI.")


>>> Kaydedildi: future_demand_forecast.csv, models/demand_forecast_model.pkl, model_comparison_results.csv, envanter_verisi.csv

TUM ADIMLAR TAMAMLANDI.
